# 🎭 PoetryDuel-GPT — Colab Training

**Train a ~15M-parameter GPT model to duel Vietnamese Lục Bát poetry.**

| Step | Time |
|------|------|
| Clone repo + install | ~2 min |
| Preprocess + tokenize | ~3 min |
| Train (5000 steps) | ~60-90 min on T4, ~30-45 min on L4 |
| Generate poetry | ~1 min |

### Requirements
- Google Colab with GPU runtime (T4 is free, L4 is faster)
- The `poems_dataset.csv` must be uploaded to your Google Drive
  at `MyDrive/poetry-dual-gpt/data/poems_dataset.csv`

In [ ]:
# @title 1. Clone Repo + Install Dependencies
!git clone https://github.com/weseegod/poetry-dual-gpt.git /content/poetry-dual-gpt 2>/dev/null
%cd /content/poetry-dual-gpt
!git pull origin main

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q pandas tokenizers tqdm
!mkdir -p data tokenizer checkpoints

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# @title 2. Mount Drive + Copy Dataset
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

# Copy poems_dataset.csv from Drive to Colab VM
DRIVE_CSV = "/content/drive/MyDrive/poetry-dual-gpt/data/poems_dataset.csv"
LOCAL_CSV = "data/poems_dataset.csv"

if os.path.exists(DRIVE_CSV):
    shutil.copy(DRIVE_CSV, LOCAL_CSV)
    size_mb = os.path.getsize(LOCAL_CSV) / 1e6
    print(f"✅ Dataset copied: {size_mb:.1f} MB")
else:
    print(f"⚠️  Dataset not found at {DRIVE_CSV}")
    print("   Upload poems_dataset.csv to your Google Drive at:")
    print("   MyDrive/poetry-dual-gpt/data/poems_dataset.csv")

In [ ]:
# @title 3. Explore Dataset
%cd /content/poetry-dual-gpt
from src.dataset import explore_dataset
df = explore_dataset()

In [ ]:
# @title 4. Preprocess (Lục Bát only) + Train Tokenizer (~3 min)
%cd /content/poetry-dual-gpt

# Preprocess: convert CSV → training pairs
!python src/preprocess.py

# Train BPE tokenizer on the corpus
!python src/train_bpe.py

In [ ]:
# @title 5. Quick Test (200 steps, ~3 min) — verify everything works
%cd /content/poetry-dual-gpt
!python src/train.py --mode test

In [ ]:
# @title 6. Full Training (~60-90 min on T4, ~30-45 min on L4)
assert torch.cuda.is_available(), "⚠️  Enable GPU runtime: Runtime → Change runtime type → T4 GPU"

%cd /content/poetry-dual-gpt
!python src/train.py --mode train

In [ ]:
# @title 7. Generate Poetry + Evaluate
%cd /content/poetry-dual-gpt

# Single generation
!python src/sample.py \
    --checkpoint checkpoints/final.pt \
    --prompt "[LUC_BAT] Thân em như chẽn lúa đòng đòng," \
    --temperature 0.75 \
    --num_samples 3

# Or interactive mode (run in a separate cell)
# !python src/sample.py --checkpoint checkpoints/final.pt --interactive

In [ ]:
# @title 8. Save Checkpoint to Drive
DRIVE_CKPT = "/content/drive/MyDrive/poetry-dual-gpt/checkpoints"
!mkdir -p {DRIVE_CKPT}
!cp -v checkpoints/final.pt {DRIVE_CKPT}/
print("✅ Checkpoint saved to Drive")

# Also save best model if it exists
import os
if os.path.exists("checkpoints/best.pt"):
    !cp -v checkpoints/best.pt {DRIVE_CKPT}/
    print("✅ Best model saved to Drive")